## Step 6 — smooth building cluster labels
**# of cells in notebook:** 1

**Purpose:** Refine the building cluster assignments created in Step 5 by identifying buildings whose cluster label is inconsistent with the labels of nearby buildings. Buildings are relabeled only when the surrounding neighborhood provides strong evidence for an alternative cluster.

**Input:**

- `spatial_multivariate_clusters_new_dist.gdb` from Step 5 for each selected block
- available clustering layers:
  - `building_scmc_newdist_k2_min10`
  - `building_scmc_newdist_k3_min10`
  - `building_scmc_newdist_k4_min10`
  - `building_scmc_newdist_k5_min10`
- `CLUSTER_ID` — the original cluster assignment
- `MEM_PROB` — cluster membership probability, where available

**Output:**

Within each block folder:

- `scmc_new_dist_relabel.gdb`
- smoothed cluster layers for each available value of k:
  - `scmc_k2_relabel`
  - `scmc_k3_relabel`
  - `scmc_k4_relabel`
  - `scmc_k5_relabel`

At the base block directory:

- `scmc_new_dist_relabel_summary.csv`

The output layers retain the original building attributes and add `cluster_smooth` together with diagnostic fields describing the local neighborhood and whether a building was relabeled.

**Main logic:**

**Cell 1 — Smooth locally inconsistent cluster labels**

1. For each Step 5 clustering layer, builds a building-neighbor graph using polygon edge-to-edge distances of 30 m or less.
2. Calculates the unweighted share of neighboring buildings belonging to the building's original cluster and to each alternative cluster.
3. Relabels a building when either:
   - its original cluster represents no more than 15% of its neighbors and an alternative cluster represents at least 85%; or
   - its original cluster represents no more than 25%, an alternative represents at least 75%, and `MEM_PROB` is no greater than 0.50.
4. Writes the revised cluster value to `cluster_smooth` and records diagnostic information describing the relabel decision.
5. Writes the smoothed layers and an overall summary of how many buildings were changed.


In [ ]:
r"""
SCMC new-distance relabeling using GeoPandas / Pyogrio
=======================================================

Purpose
-------
For each block folder under:
    E:\_johannesburg\_analysis\heterogeneous_largePop_blocks

read SCMC building layers from:
    spatial_multivariate_clusters_new_dist.gdb

build a 30 m polygon-to-polygon edge-distance neighbor graph, calculate
unweighted local label shares, and relabel buildings using these rules:

    Rule 1:
        orig_share <= 0.15 AND best_alt_share >= 0.85

    OR

    Rule 2:
        orig_share <= 0.25 AND best_alt_share >= 0.75 AND MEM_PROB <= 0.50

Important changes from the previous smoothing script
----------------------------------------------------
- Uses one distance band only: 30 m.
- Uses unweighted neighbor evidence: every neighboring building has equal weight.
- Same-label connected component size is still calculated and written as a
  diagnostic field, but it is NOT used in the relabel decision.
- Nearest-neighbor label is still written as a diagnostic field, but it is NOT
  used in the relabel decision.
- Outputs are written to a new File Geodatabase in each block folder:
      scmc_new_dist_relabel.gdb

Expected input layer names
--------------------------
The script auto-detects layers named like:
    building_scmc_newdist_k2_min10
    building_scmc_newdist_k3_min10
    building_scmc_newdist_k4_min10
    building_scmc_newdist_k5_min10

Output layer names are:
    scmc_k2_relabel
    scmc_k3_relabel
    scmc_k4_relabel
    scmc_k5_relabel

Environment notes
-----------------
This script does not use ArcPy. It requires:
    geopandas
    pandas
    shapely
    pyogrio

It also requires GDAL/OpenFileGDB write support through pyogrio. If your GDAL
build cannot write File Geodatabases, the script will stop with a clear error.
"""

import os
import re
import csv
import shutil
from collections import defaultdict, deque

import geopandas as gpd
import pandas as pd
import pyogrio


# ============================================================
# USER SETTINGS
# ============================================================

base_dir = r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"

input_gdb_name = "spatial_multivariate_clusters_new_dist.gdb"
output_gdb_name = "scmc_new_dist_relabel.gdb"

# One distance band only, in meters.
distance_m = 30

# Input layer pattern. The k value is captured and used in the output name.
input_layer_regex = re.compile(r"^building_scmc_newdist_k(\d+)_min10$", re.IGNORECASE)

# Set to None to process every block folder. For testing, use e.g. ["_1978"].
block_filter = None
# block_filter = ["_1978"]

# If True, delete and rebuild each block's output geodatabase.
# This is the cleanest approach because FileGDB layer overwrite behavior varies
# across GDAL/pyogrio versions.
overwrite_output_gdb = True

# If True, skip input layers when the corresponding output layer already exists.
# Only relevant when overwrite_output_gdb = False.
skip_existing_layers = True

# Repair invalid geometries before graph building.
repair_invalid_geometries = True

# Summary CSV written to base_dir.
summary_csv = os.path.join(base_dir, "scmc_new_dist_relabel_summary.csv")


# ------------------------------------------------------------
# Relabel rules
# ------------------------------------------------------------

rule1_orig_share_max = 0.15
rule1_best_alt_share_min = 0.85

rule2_orig_share_max = 0.25
rule2_best_alt_share_min = 0.75
rule2_mem_prob_max = 0.50


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def check_openfilegdb_write_support():
    """Stop early if pyogrio/GDAL cannot write OpenFileGDB."""
    drivers = pyogrio.list_drivers()
    support = drivers.get("OpenFileGDB")

    if support is None:
        raise RuntimeError(
            "GDAL/pyogrio does not report the OpenFileGDB driver. "
            "This environment probably cannot write Esri File Geodatabases."
        )

    # pyogrio usually reports capabilities as a string containing 'r', 'w', etc.
    if "w" not in str(support).lower():
        raise RuntimeError(
            f"GDAL/pyogrio OpenFileGDB driver is available but not writable: {support}.\n"
            "Use an environment with GDAL OpenFileGDB write support, or change the "
            "script to write GeoPackage outputs instead."
        )


def list_matching_input_layers(gdb_path):
    """
    Return [(input_layer_name, output_layer_name, k_value), ...]
    for input layers matching the expected naming pattern.
    """
    layers_info = pyogrio.list_layers(gdb_path)

    matches = []
    for row in layers_info:
        layer_name = row[0]
        m = input_layer_regex.match(layer_name)
        if not m:
            continue

        k_value = int(m.group(1))
        out_layer = f"scmc_k{k_value}_relabel"
        matches.append((layer_name, out_layer, k_value))

    matches.sort(key=lambda x: x[2])
    return matches


def output_layer_exists(gdb_path, layer_name):
    if not os.path.exists(gdb_path):
        return False
    try:
        existing = [row[0] for row in pyogrio.list_layers(gdb_path)]
        return layer_name in existing
    except Exception:
        return False


def find_cluster_field(gdf):
    """
    Finds the cluster label field. In ArcGIS it may display as 'Cluster ID',
    but the true field name is usually something like CLUSTER_ID.
    """
    candidates = [
        "Cluster ID",
        "CLUSTER_ID",
        "Cluster_ID",
        "cluster_id",
        "SS_GROUP",
        "ss_group",
        "SS_GROUP_ID",
    ]

    cols = list(gdf.columns)
    lower_lookup = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand.lower() in lower_lookup:
            return lower_lookup[cand.lower()]

    for c in cols:
        cl = c.lower()
        if "cluster" in cl and "id" in cl:
            return c

    raise ValueError(f"Could not find cluster field. Available columns:\n{cols}")


def find_mem_prob_field(gdf):
    """Find MEM_PROB field if present."""
    candidates = ["MEM_PROB", "Mem_Prob", "mem_prob", "membership_probability"]
    cols = list(gdf.columns)
    lower_lookup = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand.lower() in lower_lookup:
            return lower_lookup[cand.lower()]

    for c in cols:
        cl = c.lower()
        if "mem" in cl and "prob" in cl:
            return c

    return None


def make_valid_if_needed(gdf):
    """Repair invalid geometries if requested and needed."""
    if not repair_invalid_geometries:
        return gdf

    invalid_count = int((~gdf.geometry.is_valid).sum())
    if invalid_count == 0:
        return gdf

    print(f"    Repairing {invalid_count:,} invalid geometries...")
    try:
        gdf["geometry"] = gdf.geometry.make_valid()
    except Exception:
        # Fallback for older Shapely versions.
        gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf


def build_edge_distance_graph(gdf, distance_m):
    """
    Build an undirected neighbor graph using polygon-to-polygon distance.

    Returns
    -------
    neighbors : dict
        {position_i: [(position_j, distance), ...]}

    Notes
    -----
    The graph is based on exact Shapely geometry distance after using the
    spatial index to identify candidates. This is polygon geometry distance,
    not centroid distance.
    """
    geoms = gdf.geometry.reset_index(drop=True)
    sindex = geoms.sindex
    neighbors = defaultdict(list)

    # Modern GeoPandas/Shapely path.
    try:
        pairs = sindex.query(
            geoms,
            predicate="dwithin",
            distance=distance_m,
            output_format="indices",
        )

        left_indices = pairs[0]
        right_indices = pairs[1]

        for i, j in zip(left_indices, right_indices):
            if i >= j:
                continue

            dist = geoms.iloc[i].distance(geoms.iloc[j])
            if dist <= distance_m:
                neighbors[i].append((j, dist))
                neighbors[j].append((i, dist))

    except TypeError:
        # Fallback for older GeoPandas versions.
        for i, geom in enumerate(geoms):
            minx, miny, maxx, maxy = geom.bounds
            candidate_idx = list(
                sindex.intersection(
                    (
                        minx - distance_m,
                        miny - distance_m,
                        maxx + distance_m,
                        maxy + distance_m,
                    )
                )
            )

            for j in candidate_idx:
                if i >= j:
                    continue

                dist = geom.distance(geoms.iloc[j])
                if dist <= distance_m:
                    neighbors[i].append((j, dist))
                    neighbors[j].append((i, dist))

    return neighbors


def compute_same_label_components(labels, neighbors):
    """
    Compute same-label connected component sizes for diagnostics only.

    This is NOT used in the relabel decision in this revised script.
    """
    n = len(labels)
    visited = set()
    comp_size = [1] * n
    comp_id = [-1] * n
    current_comp = 0

    for i in range(n):
        if i in visited:
            continue

        current_comp += 1
        label_i = labels[i]
        q = deque([i])
        visited.add(i)
        members = []

        while q:
            current = q.popleft()
            members.append(current)

            for nbr, _dist in neighbors.get(current, []):
                if nbr in visited:
                    continue
                if labels[nbr] == label_i:
                    visited.add(nbr)
                    q.append(nbr)

        size = len(members)
        for member in members:
            comp_id[member] = current_comp
            comp_size[member] = size

    return comp_id, comp_size


def unweighted_label_support(i, labels, neighbors):
    """
    Calculate unweighted local label shares for one building.

    Every neighbor inside the 30 m graph gets one equal vote. Building area and
    distance-weighting are not used. Distance is still used only to determine
    whether a feature is inside the 30 m neighbor graph and to report near_label.
    """
    counts = defaultdict(int)
    nearest_idx = None
    nearest_dist = None

    for j, dist in neighbors.get(i, []):
        label_j = labels[j]
        counts[label_j] += 1

        if nearest_dist is None or dist < nearest_dist:
            nearest_idx = j
            nearest_dist = dist

    neighbor_n = sum(counts.values())

    if neighbor_n == 0:
        return {
            "neighbor_n": 0,
            "shares": {},
            "counts": {},
            "nearest_idx": nearest_idx,
            "nearest_label": None,
            "nearest_dist": nearest_dist,
        }

    shares = {label: count / neighbor_n for label, count in counts.items()}
    nearest_label = labels[nearest_idx] if nearest_idx is not None else None

    return {
        "neighbor_n": neighbor_n,
        "shares": shares,
        "counts": dict(counts),
        "nearest_idx": nearest_idx,
        "nearest_label": nearest_label,
        "nearest_dist": nearest_dist,
    }


def safe_float(value):
    if value is None or pd.isna(value):
        return None
    try:
        return float(value)
    except Exception:
        return None


def decide_new_label(i, labels, mem_probs, neighbors, comp_size):
    """
    Revised relabel decision.

    Connected component size is diagnostic only and is not part of the decision.
    Nearest-neighbor label is diagnostic only and is not part of the decision.
    """
    orig_label = labels[i]
    mem_prob = mem_probs[i] if mem_probs is not None else None

    info = unweighted_label_support(i, labels, neighbors)
    neighbor_n = info["neighbor_n"]
    shares = info["shares"]
    counts = info["counts"]
    nearest_label = info["nearest_label"]

    orig_share = shares.get(orig_label, 0.0)

    alt_candidates = {
        label: share
        for label, share in shares.items()
        if label != orig_label
    }

    if alt_candidates:
        best_alt_label = max(alt_candidates, key=alt_candidates.get)
        best_alt_share = alt_candidates[best_alt_label]
        best_alt_count = counts.get(best_alt_label, 0)
    else:
        best_alt_label = None
        best_alt_share = 0.0
        best_alt_count = 0

    rule1_pass = (
        best_alt_label is not None
        and orig_share <= rule1_orig_share_max
        and best_alt_share >= rule1_best_alt_share_min
    )

    mem_prob_value = safe_float(mem_prob)
    rule2_pass = (
        best_alt_label is not None
        and mem_prob_value is not None
        and orig_share <= rule2_orig_share_max
        and best_alt_share >= rule2_best_alt_share_min
        and mem_prob_value <= rule2_mem_prob_max
    )

    if rule1_pass:
        new_label = best_alt_label
        was_smooth = 1
        reason = "rule1_orig_le_0_15_alt_ge_0_85"
    elif rule2_pass:
        new_label = best_alt_label
        was_smooth = 1
        reason = "rule2_orig_le_0_25_alt_ge_0_75_memprob_le_0_50"
    else:
        new_label = orig_label
        was_smooth = 0

        if neighbor_n == 0:
            reason = "no_neighbors"
        elif best_alt_label is None:
            reason = "no_alternative_label"
        elif mem_prob_value is None:
            reason = "relabel_rules_not_met_memprob_missing_for_rule2"
        else:
            reason = "relabel_rules_not_met"

    return {
        "cluster_smooth": new_label,
        "was_smooth": was_smooth,
        "smooth_reason": reason,
        "nbr_n": neighbor_n,
        "orig_share": orig_share,
        "best_alt": best_alt_label,
        "best_alt_sh": best_alt_share,
        "best_alt_n": best_alt_count,
        "same_comp_n": comp_size[i],
        "near_label": nearest_label,
    }


def process_layer(input_gdb, output_gdb, input_layer, output_layer):
    print("\nProcessing layer")
    print(f"  Input GDB:    {input_gdb}")
    print(f"  Input layer:  {input_layer}")
    print(f"  Output GDB:   {output_gdb}")
    print(f"  Output layer: {output_layer}")
    print(f"  Distance:     {distance_m} m")

    if output_layer_exists(output_gdb, output_layer):
        if skip_existing_layers:
            print("  Skipping because output layer already exists.")
            return None
        raise RuntimeError(
            f"Output layer already exists and skip_existing_layers=False: {output_layer}"
        )

    gdf = gpd.read_file(input_gdb, layer=input_layer, engine="pyogrio")

    if gdf.empty:
        print("  Empty layer. Writing empty output.")
        pyogrio.write_dataframe(gdf, output_gdb, layer=output_layer, driver="OpenFileGDB")
        return {
            "changed_count": 0,
            "total_count": 0,
            "output_gdb": output_gdb,
            "output_layer": output_layer,
        }

    gdf = make_valid_if_needed(gdf)
    gdf = gdf.reset_index(drop=True)

    cluster_field = find_cluster_field(gdf)
    mem_prob_field = find_mem_prob_field(gdf)

    print(f"  Cluster field: {cluster_field}")
    print(f"  MEM_PROB field: {mem_prob_field if mem_prob_field else 'not found'}")

    labels = gdf[cluster_field].astype(str).tolist()

    if mem_prob_field is not None:
        mem_probs = [safe_float(v) for v in gdf[mem_prob_field].tolist()]
    else:
        mem_probs = None

    print("  Building 30 m edge-distance graph...")
    neighbors = build_edge_distance_graph(gdf, distance_m)

    print("  Computing same-label components for diagnostics...")
    _comp_id, comp_size = compute_same_label_components(labels, neighbors)

    print("  Applying revised unweighted relabel rules...")
    decisions = [
        decide_new_label(
            i=i,
            labels=labels,
            mem_probs=mem_probs,
            neighbors=neighbors,
            comp_size=comp_size,
        )
        for i in range(len(gdf))
    ]

    decisions_df = pd.DataFrame(decisions)

    # Keep diagnostic columns with same names as prior output.
    gdf["cluster_orig"] = gdf[cluster_field].astype(str)
    gdf["cluster_smooth"] = decisions_df["cluster_smooth"].fillna("").astype(str)
    gdf["was_smooth"] = decisions_df["was_smooth"].astype("int16")
    gdf["smooth_reason"] = decisions_df["smooth_reason"].fillna("").astype(str)

    gdf["nbr_n"] = decisions_df["nbr_n"].astype("int32")
    gdf["orig_share"] = decisions_df["orig_share"].astype(float)
    gdf["best_alt"] = decisions_df["best_alt"].fillna("").astype(str)
    gdf["best_alt_sh"] = decisions_df["best_alt_sh"].astype(float)
    gdf["best_alt_n"] = decisions_df["best_alt_n"].astype("int32")
    gdf["same_comp_n"] = decisions_df["same_comp_n"].astype("int32")
    gdf["near_label"] = decisions_df["near_label"].fillna("").astype(str)
    gdf["smooth_dist"] = float(distance_m)

    changed_count = int(gdf["was_smooth"].sum())
    total_count = len(gdf)

    print(f"  Changed: {changed_count:,} of {total_count:,}")

    print("  Writing output layer...")
    pyogrio.write_dataframe(
        gdf,
        output_gdb,
        layer=output_layer,
        driver="OpenFileGDB",
        layer_options={
            "TARGET_ARCGIS_VERSION": "ARCGIS_PRO_3_2_OR_LATER"
        },
    )

    return {
        "changed_count": changed_count,
        "total_count": total_count,
        "output_gdb": output_gdb,
        "output_layer": output_layer,
    }


def main():
    check_openfilegdb_write_support()

    block_folders = [
        os.path.join(base_dir, name)
        for name in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, name))
    ]
    block_folders = sorted(block_folders)

    if block_filter is not None:
        block_folders = [
            path for path in block_folders
            if os.path.basename(path) in set(block_filter)
        ]

    summary_rows = []

    for block_folder in block_folders:
        block_name = os.path.basename(block_folder)
        input_gdb = os.path.join(block_folder, input_gdb_name)
        output_gdb = os.path.join(block_folder, output_gdb_name)

        if not os.path.exists(input_gdb):
            print(f"\nSkipping {block_name}: input GDB not found")
            continue

        try:
            layer_matches = list_matching_input_layers(input_gdb)
        except Exception as e:
            print(f"\nSkipping {block_name}: could not list layers in input GDB")
            print(f"  {e}")
            summary_rows.append({
                "block": block_name,
                "input_layer": "",
                "output_layer": "",
                "distance_m": distance_m,
                "changed_count": "",
                "total_count": "",
                "changed_share": "",
                "output_gdb": output_gdb,
                "error": str(e),
            })
            continue

        if not layer_matches:
            print(f"\nSkipping {block_name}: no matching input layers found")
            continue

        if overwrite_output_gdb and os.path.exists(output_gdb):
            print(f"\nDeleting existing output GDB for {block_name}: {output_gdb}")
            shutil.rmtree(output_gdb)

        print(f"\nBlock {block_name}: found {len(layer_matches)} matching input layers")

        for input_layer, output_layer, k_value in layer_matches:
            try:
                result = process_layer(
                    input_gdb=input_gdb,
                    output_gdb=output_gdb,
                    input_layer=input_layer,
                    output_layer=output_layer,
                )

                if result is None:
                    continue

                changed_count = result["changed_count"]
                total_count = result["total_count"]
                changed_share = changed_count / total_count if total_count else 0

                summary_rows.append({
                    "block": block_name,
                    "input_layer": input_layer,
                    "output_layer": output_layer,
                    "distance_m": distance_m,
                    "changed_count": changed_count,
                    "total_count": total_count,
                    "changed_share": changed_share,
                    "output_gdb": output_gdb,
                    "error": "",
                })

            except Exception as e:
                print(f"\nERROR: {block_name} / {input_layer}")
                print(f"  {e}")

                summary_rows.append({
                    "block": block_name,
                    "input_layer": input_layer,
                    "output_layer": output_layer,
                    "distance_m": distance_m,
                    "changed_count": "",
                    "total_count": "",
                    "changed_share": "",
                    "output_gdb": output_gdb,
                    "error": str(e),
                })

    with open(summary_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "block",
            "input_layer",
            "output_layer",
            "distance_m",
            "changed_count",
            "total_count",
            "changed_share",
            "output_gdb",
            "error",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    print("\nDone.")
    print(f"Summary written to:\n{summary_csv}")


if __name__ == "__main__":
    main()
